In [24]:
import torch                                 
import torchvision                           
import torchvision.transforms as transforms  
import torchvision.datasets as datasets     
import matplotlib.pyplot as plt              
import numpy as np                           
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim                   
import random
from torch.utils.data import TensorDataset, DataLoader
from dataloader import load_digit_data, load_face_data, get_subset

device = ("cuda" if torch.cuda.is_available() else "mps" if hasattr(torch.backends, "mps") and torch.backends.mps.is_available() else "cpu")


In [25]:
X_train, y_train = load_digit_data("data/digitdata/trainingimages", "data/digitdata/traininglabels")


In [26]:
print(X_train.shape)
print(y_train.shape)


def split_training_data(percent, x, y):

    train_size = int((percent) * y_train.shape[0])
    indices = torch.randperm(x.shape[0])


    x_train_cut = x[indices[:train_size]]
    y_train_cut = y[indices[:train_size]]

    return torch.from_numpy(x_train_cut), torch.from_numpy(y_train_cut)



(5000, 784)
(5000,)


# Neural Network from Scratch:

The code below initializes weight matrices and bias vectors to create the manual neural network forward pass. The generator is set at the top of the cell for reproducibility. Note at the bottom I toggle all parameter gradients to True, for future automated training. For now, we will implemement the forward and backward propagation from scratch.

There are some things that will be done differently for classification. In class, we were told to perform BCE loss with every single unnormalized logit value and sum the losses across each example. However, I am applying categorical crossentropy loss, with the softmax function:

$$
\text{softmax}(z_i)=\frac{e^{z_i}}{\sum_{j=1}^{K} e^{z_j}}
$$

This function will take in a logit vector and turn it into a probability distribution by normalizing the values by the expression above. The result is that the only logit that contributes to the loss of the trianing example is the ground true logit associated with the training example.

In [39]:
'''
Following structure of the neural network:
- Input layer will be the flattened training vectors
- First hidden layer will be of size 784 x 1000
- Second hidden layer will be of size 1000 x 100
- final layer will be of size 100 x 10

I am creating the weight matrices below:
'''


seed = 42
generator = torch.Generator().manual_seed(seed)

hidden_layer_1 = (torch.randn(784, 1000,    generator = generator)).to(device)
b_1 = (torch.randn(1,1000,                  generator = generator)).to(device)

hidden_layer_2 = (torch.randn(1000, 100,    generator = generator)).to(device)
b_2  = (torch.randn(1,100,       generator = generator)).to(device)

output_layer = (torch.randn(100, 10,    generator = generator)).to(device)
b_3 = (torch.randn(1,10,       generator = generator)).to(device)



parameters = [hidden_layer_1, hidden_layer_2, output_layer, b_1, b_2, b_3]
param_list = [(hidden_layer_1, b_1), (hidden_layer_2, b_2), (output_layer, b_3)]

In [40]:
xc, yc = split_training_data(0.1, X_train, y_train)
xc = xc.to(device)
print(f"Size of input data before first linear transformation: {xc.shape}")

example_output = xc @ hidden_layer_1

print(f"Size of input data after first linear transformation: {example_output.shape}")

Size of input data before first linear transformation: torch.Size([500, 784])
Size of input data after first linear transformation: torch.Size([500, 1000])


In [41]:
'''
    Consider the forward pass function below. It traverses the tupled list of weight matrices and bias vectors per layer
    and performs the following operation: (f @ W) + b. This operation takes input batch of the previous layer and performs a
    linear transformation on the weight matrix, and then adds the bias vector (one bias value per neuron or row in the matrix).

    Note the activation function being ReLU, which is a squash at zero function. It is a piecewise function where y = 0 if x < 0
    and y = x otherwise. Because of this, the candidate derivative matrix for the ReLU function is just a matrix mask of 0's and 1's
    with a 1 in the derivative position of the value that was non-negative, and a 0 otherwise. Intuitively, ReLU fails to train neurons
    that have negative outputs, as the derivative in the computation graph is zero.

    The following function below also stores the post-activation derivatives of each hidden layer.
'''

def forward_with_computation_graph(p_list, x_batch):
    relu_derivatives = []
    layer_outputs = []

    f = x_batch
    f = f.to(device)
    for weights, bias in p_list[:-1]:
        pre_activation = (f @ weights) + bias
        f = F.relu(pre_activation)
        layer_outputs.append(f)

        relu_derivative = (f > 0)
        relu_derivatives.append(relu_derivative)
    output_weight, output_bias = p_list[-1]

    logits = (f @ output_weight) + output_bias
    return logits, relu_derivatives, layer_outputs


In [42]:
out, layer_derivs, layer_outputs = forward_with_computation_graph(param_list, xc)

print(out.shape)
print(len(layer_derivs))
print(len(layer_outputs))

torch.Size([500, 10])
2
2


In [43]:
#need a dataloader and training function here bra

def train_model(model_list, alpha, epochs, x_train, y_train, model_losses, train_percentage, bs):
    loss = nn.CrossEntropyLoss(reduction='mean')
    x, y = split_training_data(train_percentage, x_train, y_train)
    dataset = TensorDataset(x, y)
    train_loader = DataLoader(dataset, bs, shuffle=True)


    for epoch in range(epochs):
        epoch_losses, num_batches = 0.0, 0
        for x_batch, y_batch in train_loader:
            x_batch = x_batch.to(device)
            y_batch = y_batch.to(device)
            logits, layer_derivatives, layer_outputs = forward_with_computation_graph(model_list, x_batch)

            layer_1_post_activation_derivative, layer_2_post_activation_derivative = layer_derivatives

            layer_1_output, layer_2_output = layer_outputs

            (W1, b1), (W2, b2), (W_out, b_out) = model_list

            batch_loss = loss(logits, y_batch)

            epoch_losses += batch_loss
            num_batches += 1


            probs = F.softmax(logits, dim=1)

            one_hot = torch.zeros_like(logits)
            one_hot[torch.arange(y_batch.shape[0], device=logits.device), y_batch] = 1

            dL_dlogits = (probs - one_hot) / y_batch.shape[0]

            W_out_derivative = layer_2_output.T @ dL_dlogits
            b_out_derivative = torch.sum(dL_dlogits, dim=0)
            global_out_derivative = dL_dlogits @ W_out.T

            global_layer_2_derivative = layer_2_post_activation_derivative * global_out_derivative

            W2_derivative = layer_1_output.T @ global_layer_2_derivative
            b2_derivative = torch.sum(global_layer_2_derivative, dim=0)
            layer_1_post_activation = global_layer_2_derivative @ W2.T

            global_layer_1_derivative = layer_1_post_activation * layer_1_post_activation_derivative

            W1_derivative = x_batch.T @ global_layer_1_derivative
            b1_derivative = torch.sum(global_layer_1_derivative, dim=0)

            b_out -= (alpha * b_out_derivative)
            W_out -= alpha * W_out_derivative

            b2 -= (alpha *b2_derivative)
            W2 -= (alpha *W2_derivative)

            b1 -= (alpha *b1_derivative)
            W1 -=(alpha * W1_derivative)

        model_losses[epoch] = (epoch_losses / num_batches)
    return model_losses





In [44]:
losses = {}
eps = 10
percent = 0.5
batch = 64
learning_rate = 0.001

train_model(param_list, learning_rate, eps, X_train, y_train, losses, percent, batch)

{0: tensor(827.9869, device='mps:0'),
 1: tensor(258.5971, device='mps:0'),
 2: tensor(168.8563, device='mps:0'),
 3: tensor(131.2081, device='mps:0'),
 4: tensor(106.4360, device='mps:0'),
 5: tensor(85.5716, device='mps:0'),
 6: tensor(75.0910, device='mps:0'),
 7: tensor(63.8679, device='mps:0'),
 8: tensor(51.1744, device='mps:0'),
 9: tensor(44.2132, device='mps:0')}